# Task 2: Next-Day AAPL Close Prediction

## Problem Statement
Predict next-day Apple closing price from historical OHLCV data and engineered technical features.

## Goal
Build a leakage-aware time-series regression pipeline and evaluate predictive performance.

## Setup, Data Acquisition, and Preprocessing

We download data with error handling and create leakage-safe target/features.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

# Data download
try:
    df = yf.download("AAPL", period="5y", interval="1d", auto_adjust=False)
    if df is None or df.empty:
        raise RuntimeError("No data returned from yfinance.")
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
except Exception as exc:
    raise RuntimeError("Failed to download AAPL data.") from exc

# Feature engineering and target creation
df["Target_Next_Close"] = df["Close"].shift(-1)
df["SMA_10"] = df["Close"].rolling(window=10, min_periods=10).mean()
df["SMA_50"] = df["Close"].rolling(window=50, min_periods=50).mean()
df["Daily_Return"] = df["Close"].pct_change()

df = df.dropna().copy()
print(f"Prepared dataset shape: {df.shape}")
display(df.head())

: 

## Chronological Split and Scaling (Leakage Prevention)

No random split is used. The first 80% is training data and the last 20% is testing data.

In [ ]:
feature_cols = [
    "Open", "High", "Low", "Volume", "Close", "SMA_10", "SMA_50", "Daily_Return"
]

X = df[feature_cols]
y = df["Target_Next_Close"].squeeze()

split_index = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

test_dates = X_test.index

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training samples: {len(y_train)}")
print(f"Testing samples: {len(y_test)}")

## Model Training and Evaluation

In [ ]:
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)
preds = model.predict(X_test_scaled)

mae = mean_absolute_error(y_test, preds)
rmse = np.sqrt(mean_squared_error(y_test, preds))

print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")

## Visualization: Actual vs Predicted

In [ ]:
plt.figure(figsize=(12, 6))
plt.plot(test_dates, y_test.values, label="Actual", linewidth=2)
plt.plot(test_dates, preds, label="Predicted", linewidth=2, alpha=0.85)
plt.title("AAPL Next-Day Closing Price: Actual vs Predicted")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.xticks(rotation=45)
plt.legend()
plt.tight_layout()
plt.show()

## Results and Final Insights

- The workflow avoids leakage by preserving time order in splitting and fitting scaler on train only.
- MAE/RMSE quantify average and large-error behavior; compare these with baseline methods for practical usefulness.
- Next-day close prediction is inherently noisy, so this notebook should be treated as a robust baseline pipeline.
- Further gains may require richer features and return-based targets.